# Teste de Chamada LLM com LiteLLM e Groq

Este notebook demonstra como realizar chamadas de Chat/Completion usando a biblioteca `litellm` conectada à API da Groq.

Vamos utilizar os prompts definidos anteriormente (exemplo: Receita) para estruturar a resposta.

In [1]:
# !pip install litellm

In [2]:
import os
import sys
from dotenv import load_dotenv
import json
from litellm import completion
import pandas as pd

# Adicionar diretório pai
sys.path.append("..")
from src.models import Receita

# Carregar variáveis de ambiente
load_dotenv(os.path.join("..", ".env"))

# Verificar chave
if not os.environ.get("GROQ_API_KEY"):
    print("⚠️ GROQ_API_KEY não encontrada.")
else:
    print("✅ API Key carregada.")

✅ API Key carregada.


## Carregar Dados de Exemplo

Vamos carregar a transcrição da receita para usar como contexto no prompt.

In [3]:
caminho_transcricao = "../data/transcricao_receita.txt"

if os.path.exists(caminho_transcricao):
    # Assumindo que o arquivo pode ser lido diretante ou via pandas como no exemplo anterior
    # No exemplo anterior usava pd.read_csv com delimitador ';'. Vamos manter a leitura simples do texto se possível, 
    # mas se o formato for CSV, mantemos pandas.
    try:
        # Tentar ler como texto puro primeiro para ver o conteúdo
        with open(caminho_transcricao, "r", encoding="utf-8") as f:
            TEXTO_TRANSCRICAO = f.read()
        print(f"Transcrição carregada ({len(TEXTO_TRANSCRICAO)} caracteres).")
    except Exception as e:
        print(f"Erro ao ler arquivo: {e}")
else:
    print("⚠️ Arquivo de transcrição não encontrado. Usando texto de exemplo.")
    TEXTO_TRANSCRICAO = "(Texto de exemplo da receita de Zabaione...)"

Transcrição carregada (8115 caracteres).


## Definição dos Prompts

Aqui definimos o System Prompt (Persona e Instruções) e o User Prompt (Entrada de dados).

In [4]:
prompt_sistema = """
<SYSTEM>
<PERSONA>
Você é um chef de cozinha profissional, especialista em culinária internacional e criação de receitas detalhadas e muito bem explicadas. 
Você escreve sempre de forma clara, organizada e acessível.
</PERSONA>

<TAREFA>
Gerar receitas completas, detalhadas e otimizadas para leitura.
</TAREFA>

<INSTRUCOES>
- Sempre siga exatamente a transcrição fornecida pelo usuário.
- Nunca invente informações que contradigam a transcrição.
- A receita final deve ser altamente estruturada.
- Sempre responder em JSON válido.
</INSTRUCOES>

<FORMATO_RESPOSTA>
A resposta deve ser estritamente neste formato JSON:
{
  "nome_receita": "",
  "descricao": "",
  "ingredientes": [
    { "nome": "", "quantidade": "" }
  ],
  "modo_preparo": [
    {
      "passo": "",
      "instruções": ""
    }
  ],
  "tempo_preparo": "",
  "rendimento": "",
  "observacoes": ""
}
</FORMATO_RESPOSTA>
</SYSTEM>
"""

prompt_usuario = f"""
<TRANSCRIÇÃO A SEGUIR>
{TEXTO_TRANSCRICAO}
"""

## Chamada com LiteLLM

Usando o modelo `groq/llama-3.1-8b-instant`.

In [5]:
try:
    response = completion(
        model="groq/llama-3.1-8b-instant", 
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=0.2
    )
    
    resultado = response.choices[0].message.content
    print("--- Resposta Bruta ---\n")
    print(resultado)
    
except Exception as e:
    print(f"Erro na chamada LiteLLM: {e}")

--- Resposta Bruta ---

Aqui está a receita do Zabaione em formato JSON:

{
  "nome_receita": "Zabaione",
  "descricao": "Um doce italiano elaborado com gemas, açúcar e vinho licoroso, servido em taças ou copinhos.",
  "ingredientes": [
    { "nome": "Gemas de ovo", "quantidade": "6" },
    { "nome": "Açúcar", "quantidade": "40g" },
    { "nome": "Vinho licoroso (Marsala)", "quantidade": "2 colheres de sopa" }
  ],
  "modo_preparo": [
    {
      "passo": "Prepare o banho Maria e coloque a panela com água para ferver.",
      "instruções": "Coloque a panela com água para ferver e prepare o banho Maria."
    },
    {
      "passo": "Bata as gemas com o açúcar e o vinho licoroso.",
      "instruções": "Bata as gemas com o açúcar e o vinho licoroso em um fouet até que fiquem bem misturados e aerados."
    },
    {
      "passo": "Coloque a mistura no banho Maria e cozinhe por cerca de 10 minutos.",
      "instruções": "Coloque a mistura no banho Maria e cozinhe por cerca de 10 minutos, me

c:\Users\HugoLeonardo\Documents\GitHub\Data-Science---Python\Mentoria_NLP\projeto_uv\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 5: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='Aqui est...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...r_specific_fields=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
c:\Users\HugoLeonardo\Documents\GitHub\Data-Science---Python\Mentoria_NLP\projeto_uv\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 5: Expected `Message` - serialized value may not be as expected [field_name='m

## Processamento da Resposta (Pydantic)

In [6]:
# Processamento da resposta com Pydantic
if 'resultado' in globals() and resultado:
    texto_json = resultado.strip()
    
    # Remover blocos de código markdown se houver
    if texto_json.startswith("```json"):
        texto_json = texto_json.replace("```json", "", 1)
    if texto_json.startswith("```"):
        texto_json = texto_json.replace("```", "", 1)
    if texto_json.endswith("```"):
        texto_json = texto_json[:-3]
        
    texto_json = texto_json.strip()
    
    # Tentar encontrar o primeiro { e o último } para garantir
    inicio = texto_json.find("{")
    fim = texto_json.rfind("}") + 1
    
    if inicio != -1 and fim != -1:
        json_str = texto_json[inicio:fim]
        try:
            # 1. Carregar JSON puro
            dados_dict = json.loads(json_str)
            
            # 2. Validar com Pydantic
            receita_validada = Receita(**dados_dict)
            
            print("✅ Validação Pydantic: Sucesso!\n")
            
            print("--- Objeto Receita ---")
            # dump_model para dict
            print(json.dumps(receita_validada.model_dump(by_alias=True), indent=2, ensure_ascii=False))
            
            # Usar os dados validados
            df_ingredientes = pd.DataFrame([i.model_dump() for i in receita_validada.ingredientes])
            print("\n--- DataFrame de Ingredientes (Validado) ---")
            print(df_ingredientes)
                 
        except json.JSONDecodeError as je:
            print(f"❌ Erro ao decodificar JSON: {je}")
            print("Texto tentado:", json_str)
        except Exception as e:
            print(f"❌ Erro de Validação Pydantic: {e}")
    else:
        print("Não foi possível encontrar um bloco JSON válido na resposta.")
        print("Resposta original:", resultado)
else:
    print("A variável 'resultado' não está definida. Execute a célula de chamada ao modelo antes.")

✅ Validação Pydantic: Sucesso!

--- Objeto Receita ---
{
  "nome_receita": "Zabaione",
  "descricao": "Um doce italiano elaborado com gemas, açúcar e vinho licoroso, servido em taças ou copinhos.",
  "ingredientes": [
    {
      "nome": "Gemas de ovo",
      "quantidade": "6"
    },
    {
      "nome": "Açúcar",
      "quantidade": "40g"
    },
    {
      "nome": "Vinho licoroso (Marsala)",
      "quantidade": "2 colheres de sopa"
    }
  ],
  "modo_preparo": [
    {
      "passo": "Prepare o banho Maria e coloque a panela com água para ferver.",
      "instruções": "Coloque a panela com água para ferver e prepare o banho Maria."
    },
    {
      "passo": "Bata as gemas com o açúcar e o vinho licoroso.",
      "instruções": "Bata as gemas com o açúcar e o vinho licoroso em um fouet até que fiquem bem misturados e aerados."
    },
    {
      "passo": "Coloque a mistura no banho Maria e cozinhe por cerca de 10 minutos.",
      "instruções": "Coloque a mistura no banho Maria e cozinh

> streamlit para receber url e mostrar receita estruturada